In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
"""
Wine Quality Prediction - Classification Model Comparison
Task: Train and compare Random Forest, SGD, and SVC models to predict wine quality
"""

# ============================================================================
# 1. IMPORTS & SETUP
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, 
                             confusion_matrix, roc_auc_score, f1_score)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# 2. LOAD & EXPLORE DATASET
# ============================================================================
print("=" * 80)
print("STEP 1: LOADING & EXPLORING WINE QUALITY DATASET")
print("=" * 80)

# Download from UCI or use local file
# Dataset: https://archive.ics.uci.edu/dataset/186/wine+quality
# Alternative: https://www.kaggle.com/datasets/uciml/red-wine-quality

try:
    # Try loading from URL (red wine)
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    df = pd.read_csv(url, sep=';')
    print(f"✓ Dataset loaded from UCI repository")
except:
    print("Note: If offline, download from:")
    print("  - https://archive.ics.uci.edu/dataset/186/wine+quality")
    print("  - https://www.kaggle.com/datasets/uciml/red-wine-quality")
    df = pd.DataFrame()

if not df.empty:
    print(f"\nDataset Shape: {df.shape}")
    print(f"\nFirst 5 rows:")
    print(df.head())
    print(f"\nData Types:\n{df.dtypes}")
    print(f"\nMissing Values:\n{df.isnull().sum().sum()} total missing values")
    
    # ========================================================================
    # 3. CLASS DISTRIBUTION ANALYSIS
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 2: CLASS DISTRIBUTION ANALYSIS")
    print("=" * 80)
    
    print(f"\nOriginal Quality Score Distribution:")
    quality_counts = df['quality'].value_counts().sort_index()
    print(quality_counts)
    print(f"\nClass Distribution (%):")
    print((quality_counts / len(df) * 100).round(2))
    
    # Visualize original distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Count plot
    df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Distribution of Wine Quality Scores (Original)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Quality Score')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)
    
    # Percentage plot
    (quality_counts / len(df) * 100).plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Class Distribution (%)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Quality Score')
    axes[1].set_ylabel('Percentage (%)')
    axes[1].tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.savefig('./01_quality_distribution.png', dpi=300, bbox_inches='tight')
    print("\n✓ Saved: 01_quality_distribution.png")
    plt.close()
    
    # ========================================================================
    # 4. EXPLORATORY DATA ANALYSIS (EDA)
    # ============================================================================
    print("\n" + "=" * 80)
    print("STEP 3: EXPLORATORY DATA ANALYSIS (EDA)")
    print("=" * 80)
    
    # Statistical summary
    print("\nStatistical Summary:")
    print(df.describe().round(3))
    
    # Distribution plots for chemical features
    features = df.columns.drop('quality')
    n_features = len(features)
    
    fig, axes = plt.subplots(n_features // 3 + 1, 3, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, feature in enumerate(features):
        axes[idx].hist(df[feature], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
        axes[idx].set_title(f'Distribution of {feature}', fontsize=10, fontweight='bold')
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel('Frequency')
    
    # Hide empty subplots
    for idx in range(len(features), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('./02_feature_distributions.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: 02_feature_distributions.png")
    plt.close()
    
    # Correlation heatmap
    print("\nCalculating correlation matrix...")
    corr_matrix = df.corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Heatmap: Wine Quality Features', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('./03_correlation_heatmap.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: 03_correlation_heatmap.png")
    plt.close()
    
    # Top correlations with quality
    print("\nTop 10 Features Correlated with Quality:")
    quality_corr = corr_matrix['quality'].sort_values(ascending=False)
    print(quality_corr)
    
    # ========================================================================
    # 5. CLASS IMBALANCE ANALYSIS & FEATURE ENGINEERING
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 4: CLASS IMBALANCE & FEATURE ENGINEERING")
    print("=" * 80)
    
    print("\n📊 CLASS IMBALANCE ANALYSIS:")
    print("-" * 50)
    
    imbalance_ratio = quality_counts.max() / quality_counts.min()
    print(f"Imbalance Ratio (max/min): {imbalance_ratio:.2f}x")
    print(f"Most common class: Quality {quality_counts.idxmax()} ({quality_counts.max()} samples)")
    print(f"Least common class: Quality {quality_counts.idxmin()} ({quality_counts.min()} samples)")
    
    print("\n⚠️ IMPACT OF CLASS IMBALANCE:")
    print("  • Models may be biased towards majority classes")
    print("  • Minority class recall may be poor")
    print("  • Accuracy alone is not a reliable metric")
    print("  • Weighted metrics and stratified sampling are essential")
    
    # Feature Engineering: Binning quality scores
    print("\n🔧 FEATURE ENGINEERING - BINNING QUALITY SCORES:")
    print("-" * 50)
    
    # Create multiple target variables for comparison
    # Binary: Good (≥6) vs Bad (<6)
    df['quality_binary'] = (df['quality'] >= 6).astype(int)
    
    # 3-class: Low (≤4), Medium (5-6), High (≥7)
    df['quality_3class'] = pd.cut(df['quality'], 
                                   bins=[0, 4, 6, 10], 
                                   labels=['Low', 'Medium', 'High'],
                                   include_lowest=True)
    
    print("\nOption 1: Binary Classification (Good ≥6 vs Bad <5)")
    print(df['quality_binary'].value_counts())
    print(f"Distribution: {(df['quality_binary'].value_counts(normalize=True) * 100).round(2).to_dict()}")
    
    print("\nOption 2: 3-Class Classification (Low ≤4, Medium 5-6, High ≥7)")
    print(df['quality_3class'].value_counts())
    print(f"Distribution: {(df['quality_3class'].value_counts(normalize=True) * 100).round(2).to_dict()}")
    
    print("\n✅ DECISION: Using Binary Classification (Good/Bad)")
    print("   Rationale:")
    print("   - Better balanced distribution")
    print("   - Simpler interpretation for deployment")
    print("   - Reduces noise from borderline quality scores")
    
    target = 'quality_binary'
    
    # Visualize class distribution for selected target
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Original quality
    df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Original Quality Distribution', fontweight='bold')
    axes[0].set_xlabel('Quality Score')
    axes[0].set_ylabel('Count')
    
    # Binary
    df['quality_binary'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Binary Classification (Good/Bad)', fontweight='bold')
    axes[1].set_xticklabels(['Bad (<6)', 'Good (≥6)'], rotation=0)
    axes[1].set_ylabel('Count')
    
    # 3-class
    df['quality_3class'].value_counts().plot(kind='bar', ax=axes[2], color='lightgreen')
    axes[2].set_title('3-Class Classification', fontweight='bold')
    axes[2].set_ylabel('Count')
    
    plt.tight_layout()
    plt.savefig('./04_feature_engineering.png', dpi=300, bbox_inches='tight')
    print("\n✓ Saved: 04_feature_engineering.png")
    plt.close()
    
    # ========================================================================
    # 6. DATA PREPARATION & TRAIN/TEST SPLIT
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 5: DATA PREPARATION & TRAIN/TEST SPLIT")
    print("=" * 80)
    
    # Prepare features and target
    X = df[features]
    y = df[target]
    
    print(f"\nFeature Matrix Shape: {X.shape}")
    print(f"Target Shape: {y.shape}")
    
    # Train/test split with stratification
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"\nTrain Set Size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
    print(f"Test Set Size: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
    
    print(f"\nTrain Set Class Distribution:")
    print(y_train.value_counts())
    print(f"\nTest Set Class Distribution:")
    print(y_test.value_counts())
    
    # Feature scaling (needed for SGD and SVC)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print("\n✓ Features scaled using StandardScaler")
    
    # ========================================================================
    # 7. MODEL TRAINING
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 6: TRAINING CLASSIFIERS")
    print("=" * 80)
    
    models = {
        'Random Forest': RandomForestClassifier(
            n_estimators=100, 
            max_depth=10, 
            random_state=42, 
            n_jobs=-1,
            class_weight='balanced'
        ),
        'SGD': SGDClassifier(
            loss='log_loss',  # logistic regression
            random_state=42,
            n_jobs=-1,
            class_weight='balanced',
            max_iter=1000
        ),
        'SVC': SVC(
            kernel='rbf',
            random_state=42,
            class_weight='balanced',
            probability=True
        )
    }
    
    trained_models = {}
    
    for name, model in models.items():
        print(f"\n🔄 Training {name}...")
        
        if name == 'Random Forest':
            model.fit(X_train, y_train)
        else:
            model.fit(X_train_scaled, y_train)
        
        trained_models[name] = model
        print(f"✓ {name} trained successfully")
    
    # ========================================================================
    # 8. MODEL EVALUATION
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 7: MODEL EVALUATION")
    print("=" * 80)
    
    results = {}
    
    for name, model in trained_models.items():
        print(f"\n{'='*60}")
        print(f"MODEL: {name}")
        print(f"{'='*60}")
        
        # Make predictions
        if name == 'Random Forest':
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1]
        else:
            y_pred = model.predict(X_test_scaled)
            y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        
        results[name] = {
            'Accuracy': accuracy,
            'F1-Score': f1,
            'ROC-AUC': roc_auc,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba,
            'conf_matrix': confusion_matrix(y_test, y_pred)
        }
        
        print(f"\nAccuracy: {accuracy:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print(f"ROC-AUC Score: {roc_auc:.4f}")
        
        print(f"\nClassification Report:")
        print(classification_report(y_test, y_pred, 
                                   target_names=['Bad (<6)', 'Good (≥6)']))
    
    # ========================================================================
    # 9. CONFUSION MATRICES
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 8: CONFUSION MATRICES")
    print("=" * 80)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    for idx, (name, metrics) in enumerate(results.items()):
        cm = metrics['conf_matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                   cbar=False, xticklabels=['Bad', 'Good'], yticklabels=['Bad', 'Good'])
        axes[idx].set_title(f'{name}\nAccuracy: {metrics["Accuracy"]:.4f}', 
                           fontweight='bold', fontsize=11)
        axes[idx].set_ylabel('True Label')
        axes[idx].set_xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.savefig('./05_confusion_matrices.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: 05_confusion_matrices.png")
    plt.close()
    
    # ========================================================================
    # 10. FEATURE IMPORTANCE (Random Forest)
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 9: FEATURE IMPORTANCE ANALYSIS")
    print("=" * 80)
    
    rf_model = trained_models['Random Forest']
    feature_importance = pd.DataFrame({
        'Feature': features,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nRandom Forest Feature Importance:")
    print(feature_importance)
    
    # Plot top 11 features
    top_features = feature_importance.head(11)
    
    plt.figure(figsize=(10, 6))
    bars = plt.barh(range(len(top_features)), top_features['Importance'], color='steelblue')
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.xlabel('Importance Score', fontweight='bold')
    plt.title('Top 11 Most Important Features - Random Forest', fontweight='bold', fontsize=12)
    
    # Add value labels on bars
    for i, bar in enumerate(bars):
        width = bar.get_width()
        plt.text(width, bar.get_y() + bar.get_height()/2, 
                f'{width:.4f}', ha='left', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('./06_feature_importance.png', dpi=300, bbox_inches='tight')
    print("\n✓ Saved: 06_feature_importance.png")
    plt.close()
    
    # ========================================================================
    # 11. MODEL COMPARISON
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 10: MODEL COMPARISON")
    print("=" * 80)
    
    comparison_df = pd.DataFrame({
        'Model': list(results.keys()),
        'Accuracy': [results[m]['Accuracy'] for m in results.keys()],
        'F1-Score': [results[m]['F1-Score'] for m in results.keys()],
        'ROC-AUC': [results[m]['ROC-AUC'] for m in results.keys()]
    }).round(4)
    
    print("\n📊 PERFORMANCE COMPARISON TABLE:")
    print(comparison_df.to_string(index=False))
    
    # Save comparison table
    comparison_df.to_csv('./model_comparison.csv', index=False)
    print("\n✓ Saved: model_comparison.csv")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    metrics = ['Accuracy', 'F1-Score', 'ROC-AUC']
    colors = ['steelblue', 'coral', 'lightgreen']
    
    for idx, metric in enumerate(metrics):
        values = comparison_df[metric].values
        bars = axes[idx].bar(comparison_df['Model'], values, color=colors[idx], alpha=0.7, edgecolor='black')
        axes[idx].set_title(f'{metric} Comparison', fontweight='bold', fontsize=12)
        axes[idx].set_ylabel(metric, fontweight='bold')
        axes[idx].set_ylim([0, 1])
        axes[idx].tick_params(axis='x', rotation=15)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                          f'{height:.4f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('./07_model_comparison.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: 07_model_comparison.png")
    plt.close()
    
    # ========================================================================
    # 12. CONCLUSION & RECOMMENDATIONS
    # ========================================================================
    print("\n" + "=" * 80)
    print("STEP 11: CONCLUSION & DEPLOYMENT RECOMMENDATIONS")
    print("=" * 80)
    
    best_model_idx = comparison_df['Accuracy'].idxmax()
    best_model_name = comparison_df.loc[best_model_idx, 'Model']
    best_accuracy = comparison_df.loc[best_model_idx, 'Accuracy']
    best_f1 = comparison_df.loc[best_model_idx, 'F1-Score']
    best_roc_auc = comparison_df.loc[best_model_idx, 'ROC-AUC']
    
    print(f"\n🏆 BEST PERFORMING MODEL: {best_model_name}")
    print(f"   • Accuracy: {best_accuracy:.4f}")
    print(f"   • F1-Score: {best_f1:.4f}")
    print(f"   • ROC-AUC: {best_roc_auc:.4f}")
    
    print("\n" + "="*60)
    print("DETAILED ANALYSIS & RECOMMENDATIONS")
    print("="*60)
    
    print("\n1️⃣ RANDOM FOREST:")
    print(f"   Accuracy: {comparison_df.loc[comparison_df['Model']=='Random Forest', 'Accuracy'].values[0]:.4f}")
    print("   ✅ Strengths:")
    print("      - Best for feature interpretation (feature importance available)")
    print("      - Handles non-linear relationships well")
    print("      - No feature scaling required")
    print("   ❌ Weaknesses:")
    print("      - Larger memory footprint and slower predictions")
    print("      - Risk of overfitting with high tree count")
    
    print("\n2️⃣ STOCHASTIC GRADIENT DESCENT (SGD):")
    print(f"   Accuracy: {comparison_df.loc[comparison_df['Model']=='SGD', 'Accuracy'].values[0]:.4f}")
    print("   ✅ Strengths:")
    print("      - Fast training and predictions")
    print("      - Efficient for large datasets")
    print("      - Good for online learning scenarios")
    print("   ❌ Weaknesses:")
    print("      - Requires feature scaling")
    print("      - Less stable than ensemble methods")
    print("      - Hyperparameter tuning critical")
    
    print("\n3️⃣ SUPPORT VECTOR CLASSIFIER (SVC):")
    print(f"   Accuracy: {comparison_df.loc[comparison_df['Model']=='SVC', 'Accuracy'].values[0]:.4f}")
    print("   ✅ Strengths:")
    print("      - Excellent for binary classification")
    print("      - Robust to outliers")
    print("      - Works well in high-dimensional spaces")
    print("   ❌ Weaknesses:")
    print("      - Slower training on large datasets")
    print("      - Requires feature scaling")
    print("      - Less interpretable than Random Forest")
    
    print("\n" + "="*60)
    print("FINAL RECOMMENDATION FOR DEPLOYMENT")
    print("="*60)
    
    print(f"\n🚀 RECOMMENDED MODEL: {best_model_name}")
    print("\nRationale:")
    if best_model_name == 'Random Forest':
        print("  1. Highest overall accuracy among the three models")
        print("  2. Built-in feature importance for model interpretability")
        print("  3. Robust to outliers and handles non-linear patterns")
        print("  4. No feature scaling required (simpler preprocessing)")
        print("  5. Balanced class weights handle imbalanced data effectively")
    else:
        print(f"  1. Superior performance metrics (Accuracy: {best_accuracy:.4f})")
        print(f"  2. Best F1-Score: {best_f1:.4f} (balanced precision-recall)")
        print(f"  3. Best ROC-AUC: {best_roc_auc:.4f} (discrimination ability)")
        print("  4. Efficient for production deployment")
    
    print("\nDeployment Checklist:")
    print("  ☐ Save trained model using joblib or pickle")
    print("  ☐ Save feature scaler for SGD/SVC if needed")
    print("  ☐ Create prediction API endpoint")
    print("  ☐ Implement model monitoring for drift detection")
    print("  ☐ Set up retraining pipeline for periodic updates")
    print("  ☐ Document feature engineering and preprocessing steps")
    print("  ☐ Create unit tests for prediction consistency")
    
    print("\n" + "=" * 80)
    print("✅ ANALYSIS COMPLETE - All outputs saved to ./")
    print("=" * 80)

else:
    print("\n⚠️ Dataset not loaded. Please download from:")
    print("   https://archive.ics.uci.edu/dataset/186/wine+quality")
    print("   or https://www.kaggle.com/datasets/uciml/red-wine-quality")

STEP 1: LOADING & EXPLORING WINE QUALITY DATASET
✓ Dataset loaded from UCI repository

Dataset Shape: (1599, 12)

First 5 rows:
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0           